In [4]:
import pandas as pd
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

# Load Dataset
file_path = "Roman-Urdu-Poetry.csv"
actual_df = pd.read_csv(file_path)

# Keep only the Poetry column
df = actual_df[["Poetry"]].dropna()

# Function to clean text
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^a-zA-ZāčēğīñōūṣṭẓḳḌ -]', '', text)  # Remove special characters except space
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

# Apply text cleaning
df["Poetry"] = df["Poetry"].apply(clean_text)
# Show some cleaned data


In [5]:
# Tokenization function
def tokenize_texts(texts, oov_token="<OOV>"):
    tokenizer = Tokenizer(oov_token=oov_token)
    tokenizer.fit_on_texts(texts)
    sequences = tokenizer.texts_to_sequences(texts)
    return tokenizer, sequences

# Function to prepare input-output pairs using sliding window approach
def generate_training_data(sequences, sequence_length=10):
    input_sequences = []
    output_words = []
    for seq in sequences:
        for i in range(1, len(seq)):
            input_sequences.append(seq[max(0, i-sequence_length):i]) #Take all elements from the starting index up to (but not including) i(predicted word)
            output_words.append(seq[i])

    return input_sequences, output_words

# Function to pad sequences
def pad_input_sequences(input_sequences):
    max_sequence_length = max(len(seq) for seq in input_sequences)
    padded_sequences = pad_sequences(input_sequences, maxlen=max_sequence_length, padding='pre')
    return padded_sequences, max_sequence_length

# Main processing pipeline
def preprocess_data(texts, sequence_length=10):
    tokenizer, sequences = tokenize_texts(texts)
    input_sequences, output_words = generate_training_data(sequences, sequence_length)
    padded_sequences, max_seq_length = pad_input_sequences(input_sequences)
    return tokenizer, padded_sequences, np.array(output_words), max_seq_length

# Run preprocessing
tokenizer, input_sequences, output_words, max_seq_length = preprocess_data(df["Poetry"].tolist())



In [6]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Define the LSTM Model
vocab_size = len(tokenizer.word_index) + 1  # vocab size from tokenizer
embedding_dim = 150

model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_seq_length),  # input_length = max sequence length
    LSTM(256, return_sequences=False),  # LSTM layer
    Dropout(0.4),  # Dropout layer to avoid overfitting
    Dense(vocab_size, activation='softmax', kernel_regularizer=tf.keras.regularizers.l2(0.01))  # Output layer with softmax activation
])

# Define a learning rate scheduler
def lr_scheduler(epoch, lr):
    new_lr = lr if epoch < 10 else lr * tf.math.exp(-0.1)
    return float(new_lr)  # Make sure the returned value is a float

# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
checkpoint = ModelCheckpoint("lstm_poetry_model.keras", save_best_only=True, monitor='val_loss')

# Preprocess input and output
input_sequences = np.array(input_sequences)  # Convert to NumPy array
# Ensure the output_words is correctly derived from the last word of each sequence
output_words = np.array([seq[-1] for seq in input_sequences])  # Extract last word of each sequence

# Debugging prints
print(f"Input shape: {input_sequences.shape}")  # Expected: (batch_size, max_seq_length)
print(f"Output shape: {output_words.shape}")  # Expected: (batch_size,)

# Pad sequences for consistency
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_length, padding='pre')

# Compile the model
model.compile(loss='sparse_categorical_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])

# Train the model
model.fit(input_sequences, output_words, batch_size=32, epochs=5, validation_split=0.2,
          callbacks=[early_stopping, checkpoint, tf.keras.callbacks.LearningRateScheduler(lr_scheduler)])

# Show model summary
model.summary()


C:\Users\Ayesha\AppData\Roaming\Python\Python310\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Input shape: (179777, 10)
Output shape: (179777,)
Epoch 1/20
4495/4495 ━━━━━━━━━━━━━━━━━━━━ 1142s 252ms/step - accuracy: 0.0955 - loss: 6.8739 - val_accuracy: 0.2482 - val_loss: 6.0922 - learning_rate: 0.0010
Epoch 2/20
4495/4495 ━━━━━━━━━━━━━━━━━━━━ 1120s 249ms/step - accuracy: 0.3106 - loss: 5.6721 - val_accuracy: 0.3777 - val_loss: 5.6869 - learning_rate: 0.0010
Epoch 3/20
4495/4495 ━━━━━━━━━━━━━━━━━━━━ 1145s 255ms/step - accuracy: 0.4099 - loss: 5.2055 - val_accuracy: 0.4371 - val_loss: 5.5599 - learning_rate: 0.0010
Epoch 4/20
4495/4495 ━━━━━━━━━━━━━━━━━━━━ 1142s 250ms/step - accuracy: 0.4448 - loss: 5.0425 - val_accuracy: 0.4593 - val_loss: 5.4653 - learning_rate: 0.0010
Epoch 5/20
4401/4495 ━━━━━━━━━━━━━━━━━━━━ 24s 262ms/step - accuracy: 0.4705 - loss: 4.8968

KeyboardInterrupt: 

In [ ]:
# Function to generate text
def generate_poetry(seed_text, next_words=30):
    for _ in range(next_words):
        tokenized_input = tokenizer.texts_to_sequences([seed_text])
        tokenized_input = pad_sequences(tokenized_input, maxlen=max_seq_length, padding='pre')
        predicted_index = np.argmax(model.predict(tokenized_input), axis=-1)[0]
        predicted_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                predicted_word = word
                break
        seed_text += " " + predicted_word
    return seed_text

# Example Usage
seed_text = "ishq"
generated_poetry = generate_poetry(seed_text)
print("Generated Poetry:", generated_poetry)